# Model Training

### Importing Libraries

In [ ]:
import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns

from sklearn.ensemble import IsolationForest 

from sklearn.neighbors import LocalOutlierFactor 

from sklearn.covariance import EllipticEnvelope  

from joblib import Parallel, delayed

In [ ]:
# Load data

df = pd.read_csv(r"C:\Users\Akshay\Desktop\building-energy-anomaly-detection\data\meters\whole\eda.csv")

# Convert timestamp to datetime and sort

df['timestamp'] = pd.to_datetime(df['timestamp'])

df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Dataset shape: {df.shape}")

print(f"Columns: {df.columns.tolist()}")

### Feature Preparation

In [ ]:
# Select numeric columns for modeling (exclude timestamp)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Remove timestamp from numeric_cols if present
if 'timestamp' in numeric_cols:
    numeric_cols.remove('timestamp')

print(f"Number of numeric features: {len(numeric_cols)}")

# Prepare feature matrix
X = df[numeric_cols].fillna(0).values

print(f"Feature matrix shape: {X.shape}")

### Model Training

In [ ]:
# Train Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)
anomaly_iso = iso_forest.fit_predict(X)

# Train Local Outlier Factor
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05,
    n_jobs=-1
)
anomaly_lof = lof.fit_predict(X)

# Define and train Elliptic Envelope (Mahalanobis)
def train_elliptic_envelope(X):
    n_samples = len(X)
    n_features = X.shape[1]
    
    if n_samples < 100 or n_features > 500:
        support_fraction = 0.5
    elif n_samples < 500 or n_features > 1000:
        support_fraction = 0.6
    else:
        support_fraction = 0.7
    
    robust_cov = EllipticEnvelope(
        contamination=0.05,
        random_state=42,
        support_fraction=support_fraction
    )
    return robust_cov.fit_predict(X), robust_cov

anomaly_maha, robust_cov = train_elliptic_envelope(X)

print("3 models built successfully")
print("Anomaly detection complete!")

### Model Evaluation

In [ ]:
# Each model votes: 1 if anomaly (-1), 0 if normal (1)
df['anomaly_votes'] = ((anomaly_iso == -1).astype(int) + 
                        (anomaly_lof == -1).astype(int) + 
                        (anomaly_maha == -1).astype(int))
df['is_anomaly'] = (df['anomaly_votes'] >= 2).astype(int)

In [ ]:
# Using decision_function scores to understand anomaly severity
anomaly_scores = iso_forest.decision_function(X)

# Create a dataframe with scores
scores_df = pd.DataFrame({
    'score': anomaly_scores,
    'is_anomaly': df['is_anomaly']
})

# Plot anomaly score distribution
plt.figure(figsize=(10, 6))
plt.hist(anomaly_scores, bins=50, edgecolor="black", alpha=0.7)
plt.xlabel("Anomaly Score")
plt.ylabel("Frequency")
plt.title("Distribution of Isolation Forest Anomaly Scores")
plt.axvline(x=np.percentile(anomaly_scores, 5), color="red", linestyle="--", 
            label="5th Percentile (threshold)")
plt.legend()
plt.tight_layout()
plt.savefig("anomaly_scores_distribution.png", dpi=300)
plt.show()

In [ ]:
# Using feature means for normal vs anomalous data
feature_importance = {}

for col in numeric_cols:
    normal_mean = df[df["is_anomaly"] == 0][col].mean()
    anomaly_mean = df[df["is_anomaly"] == 1][col].mean()
    # Higher difference indicates more important feature
    feature_importance[col] = abs(anomaly_mean - normal_mean)

# Sort by importance
feature_importance = dict(sorted(feature_importance.items(), 
                                  key=lambda x: x[1], reverse=True))

plt.figure(figsize=(10, 6))
plt.barh(list(feature_importance.keys())[:10], 
         list(feature_importance.values())[:10])
plt.xlabel("Importance (Mean Difference)")  
plt.title("Feature Importance - Normal vs Anomaly")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=300)
plt.show()

In [ ]:
# Anomaly detection results
anomaly_df = df[df["is_anomaly"] == 1] 

print(f"Detected {len(anomaly_df)} anomalies ({len(anomaly_df)/len(df)*100:.1f}%)") 

# Time series visualization of anomalies
plt.figure(figsize=(14, 6)) 

# Convert timestamp to string for plotting if needed
df['timestamp_str'] = df['timestamp'].astype(str)
anomaly_df_plot = df[df["is_anomaly"] == 1]

# Plot normal data
plt.plot(range(len(df)), df["electricity"][:len(df)], 
         label="Normal", alpha=0.7)

# Plot anomalies
if len(anomaly_df_plot) > 0:
    plt.scatter(anomaly_df_plot.index, anomaly_df_plot["electricity"], 
                color="red", label="Anomaly", s=100, zorder=5)

plt.xlabel("Time Index")
plt.ylabel("Electricity")
plt.legend()
plt.title("Electricity Consumption with Detected Anomalies")
plt.tight_layout()
plt.savefig("anomaly_detection_plot.png", dpi=300)
plt.show()

In [ ]:
# Summary statistics
print("\n=== Anomaly Detection Summary ===")
print(f"Total data points: {len(df)}")
print(f"Normal points: {len(df) - len(anomaly_df)}")
print(f"Anomalous points: {len(anomaly_df)}")
print(f"\nAnomaly votes distribution:")
print(df["anomaly_votes"].value_counts().sort_index())

# Save results
df.to_csv("anomaly_detection_results.csv", index=False)

print("\nResults saved to anomaly_detection_results.csv")